In [1]:
#!/usr/bin/env python3
"""
unpack.py
---------
Copy the test bundle to the working dir, unzip it, and report what's inside.

    python -u unpack.py

Handles either zip layout:
    audio/<id>.mp3 + manual_transcript/<id>.json
    <id>.mp3 + <id>.json          (flat)

Idempotent: re-running skips the copy and the unzip if already done.
Pass force=True to redo.
"""

import json
import shutil
import zipfile
from pathlib import Path

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None


BASE = Path("/home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning")

CFG = {
    "zip_src":    Path("/mnt/03modeling/6_Kuladeep/test_manual.zip"),   # <-- adjust if needed
    "work_dir":   BASE / "test_set",
    "audio_exts": (".mp3", ".wav", ".m4a", ".webm", ".flac", ".ogg"),
}


def show(rows, headers):
    if tabulate:
        print(tabulate(rows, headers=headers, tablefmt="github"), flush=True)
    else:
        print("  ".join(headers))
        for r in rows:
            print("  ".join(str(x) for x in r))


def unpack(force=False):
    src = Path(CFG["zip_src"])
    if not src.is_absolute():
        src = Path.cwd() / src
    assert src.exists(), "zip not found: {}".format(src)

    work = CFG["work_dir"]
    if force and work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True, exist_ok=True)

    local_zip = Path.cwd() / src.name
    if not local_zip.exists():
        print("copying {} ({:.1f} MB) -> {}".format(
            src.name, src.stat().st_size / 1e6, local_zip))
        shutil.copy(src, local_zip)
    else:
        print("zip already in cwd: {}".format(local_zip))

    if not any(work.rglob("*.json")):
        print("unzipping -> {}".format(work))
        with zipfile.ZipFile(local_zip) as z:
            z.extractall(work)
    else:
        print("already unpacked: {}".format(work))

    return work


def inventory(work=None):
    work = work or CFG["work_dir"]

    audio = {p.stem: p for p in work.rglob("*")
             if p.is_file() and p.suffix.lower() in CFG["audio_exts"]}
    jsons = {p.stem: p for p in work.rglob("*.json")
             if p.name != "testset_index.json"}

    rows = []
    for stem in sorted(set(audio) | set(jsons)):
        a, j = audio.get(stem), jsons.get(stem)
        meta = {}
        if j:
            try:
                meta = json.loads(j.read_text(encoding="utf-8"))
            except Exception as e:
                meta = {"_error": str(e)[:40]}
        rows.append([
            stem[:24],
            "{:.1f}".format(a.stat().st_size / 1e6) if a else "-",
            "yes" if j else "MISSING",
            meta.get("transcript_source", "-"),
            meta.get("is_ground_truth", "-"),
            meta.get("speech_minutes", "-"),
            meta.get("n_segments", "-"),
        ])

    show(rows, ["id", "audio MB", "json", "source", "is_gt", "min", "segs"])

    paired = sorted(set(audio) & set(jsons))
    print("\n  audio files      : {}".format(len(audio)))
    print("  transcripts      : {}".format(len(jsons)))
    print("  PAIRED (usable)  : {}".format(len(paired)))

    only_a = sorted(set(audio) - set(jsons))
    only_j = sorted(set(jsons) - set(audio))
    if only_a:
        print("  audio w/o json   : {}".format(", ".join(only_a)))
    if only_j:
        print("  json w/o audio   : {}".format(", ".join(only_j)))

    return paired


if __name__ == "__main__":
    work = unpack()
    print()
    inventory(work)

copying test_manual.zip (53.5 MB) -> /mnt/batch/tasks/shared/LS_root/mounts/clusters/computer-vision-team-i2/code/Users/kuladeep.a/STT_finetuning/test_manual.zip
unzipping -> /home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning/test_set

| id          |   audio MB | json   | source    | is_gt   |   min |   segs |
|-------------|------------|--------|-----------|---------|-------|--------|
| 67Ljjk8Blfc |        9.5 | yes    | generated | False   |  11.5 |    142 |
| Wuu3Gd4Q6Uo |       16.1 | yes    | manual    | True    |  10   |    139 |
| lLnHN0IWic0 |        7.6 | yes    | manual    | True    |   4.3 |     76 |
| mYmNM8-XRP0 |        9.8 | yes    | manual    | True    |   5.2 |    130 |
| yR6pLfgEK3U |        5.8 | yes    | manual    | True    |   2.3 |     46 |
| zQvFSGxDiBI |        5   | yes    | manual    | True    |   3   |     40 |

  audio files      : 6
  transcripts      : 6
  PAIRED (usable)  : 6


In [4]:
#!/usr/bin/env python3
"""
wer_eval.py
-----------
Per-file WER + global averages for baseline turbo and ANY number of LoRA
checkpoints across multiple runs (rank sweeps).

    tmux new -s wer
    conda activate agentic_env
    python -u wer_eval.py 2>&1 | tee logs/wer_$(date +%m%d_%H%M).log

----------------------------------------------------------------------------
REFERENCE TIERS ARE SCORED SEPARATELY AND NEVER POOLED
----------------------------------------------------------------------------
  VERIFIED    manual caption track, is_ground_truth=True, no segment overlap.
              This is the headline number.
  UNVERIFIED  auto-caption (rolling-window overlap) or Whisper-generated.
              Scoring against these measures AGREEMENT WITH ANOTHER ASR, not
              accuracy - a model is penalised for being right where the
              reference was wrong. Reported for completeness only. If a
              checkpoint wins here and loses on VERIFIED, it has learned to
              imitate the other ASR, which is the opposite of what you want.

  MICRO WER = total errors / total ref words   (headline; long files dominate)
  MACRO WER = mean of per-file WERs            (every file counts once)
----------------------------------------------------------------------------
"""

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import gc
import json
import re
import time
import unicodedata
from pathlib import Path

import torch
import jiwer
from peft import PeftModel
from transformers import (WhisperForConditionalGeneration, WhisperProcessor,
                          pipeline)
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None


BASE = Path("/home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning")

CFG = {
    "model":    "openai/whisper-large-v3-turbo",
    "test_dir": BASE / "test_set",
    "out_dir":  BASE / "eval_out",

    # --- CHECKPOINTS ------------------------------------------------------
    # Each entry is either:
    #   a RUN DIR   -> every checkpoint-* inside is discovered and evaluated
    #   a CHECKPOINT DIR -> evaluated on its own
    # Tags are auto-built as "<run folder>:<step>".
    "checkpoint_paths": [
        BASE / "stt_audio" / "whisper_lora_512" / "checkpoint-40",       # run dir: all ckpts
        BASE / "stt_audio" / "whisper_lora_64",
        BASE / "stt_audio" / "whisper_lora" ,   # single
    ],
    "only_steps": None,          # e.g. [40, 180, 200] to filter; None = all
    # ----------------------------------------------------------------------

    "audio_exts": (".mp3", ".wav", ".m4a", ".webm", ".flac", ".ogg"),
    "language":   "en",
    "task":       "transcribe",
    "long_form":  "chunked",     # "chunked" (fast) | "sequential" (Whisper native, more accurate)
    "chunk_len":  30,
    "batch_size": 4,
    "score_unverified": True,    # score them, but in their own bucket

    # optional: train manifest(s) to check for test-set leakage
    "train_manifests": [
        BASE / "stt_audio" / "ft_dataset" / "train.jsonl",
    ],
}

GLOSSARY = [
    "quarter panel", "fender", "bumper", "bumper cover", "bonnet", "hood",
    "unibody", "chassis", "radiator", "headlight", "tail light", "windshield",
    "airbag", "frame damage", "structural damage", "rocker panel", "dent",
    "crease", "scratch", "scuff", "body filler", "primer", "sand", "panel",
    "paint", "salvage title", "write off", "total loss", "deductible",
    "adjuster", "appraisal", "estimate", "oem", "aftermarket", "axle",
    "rear ended", "paintless dent repair",
]

normalizer = EnglishTextNormalizer({})


def show(rows, headers):
    if tabulate:
        print(tabulate(rows, headers=headers, tablefmt="github"), flush=True)
    else:
        print("  ".join(headers))
        for r in rows:
            print("  ".join(str(x) for x in r))


# --------------------------------------------------------------- TEXT PREP
def norm(t):
    t = unicodedata.normalize("NFKC", t)
    t = t.replace("-", " ")
    return re.sub(r"\s+", " ", normalizer(t)).strip()


NORM_TERMS = [(t, norm(t)) for t in GLOSSARY]


def term_counts(text):
    c = {}
    for raw, nt in NORM_TERMS:
        n = len(re.findall(r"\b" + re.escape(nt) + r"\b", text))
        if n:
            c[raw] = n
    return c


def entity_recall(ref, hyp):
    rc, hc = term_counts(ref), term_counts(hyp)
    n = sum(rc.values())
    if not n:
        return None, 0, {}
    tp = sum(min(v, hc.get(k, 0)) for k, v in rc.items())
    missed = {k: v - hc.get(k, 0) for k, v in rc.items() if v > hc.get(k, 0)}
    return round(tp / n, 4), n, missed


# ------------------------------------------------------- CHECKPOINT DISCOVERY
def discover_checkpoints():
    """Expand run dirs into individual checkpoints. Tag = '<run>:<step>'."""
    found = []
    for p in CFG["checkpoint_paths"]:
        p = Path(p)
        if not p.exists():
            print("  WARNING: path does not exist, skipped: {}".format(p))
            continue
        if (p / "adapter_config.json").exists():          # a checkpoint itself
            cks = [p]
        else:                                             # a run dir
            cks = sorted(p.glob("checkpoint-*"),
                         key=lambda d: int(re.sub(r"\D", "", d.name) or 0))
            fin = p / "final"
            if fin.exists():
                cks.append(fin)
        for c in cks:
            if not (c / "adapter_config.json").exists():
                continue
            step = re.sub(r"\D", "", c.name) or c.name
            if CFG["only_steps"] and step.isdigit() and \
               int(step) not in CFG["only_steps"]:
                continue
            run = c.parent.name.replace("whisper_lora_", "r")
            found.append(("{}:{}".format(run, step), c))

    rows = []
    for tag, c in found:
        cfgp = c / "adapter_config.json"
        try:
            ac = json.loads(cfgp.read_text())
            rows.append([tag, ac.get("r"), ac.get("lora_alpha"),
                         len(ac.get("target_modules") or []), str(c.parent.name)])
        except Exception:
            rows.append([tag, "?", "?", "?", str(c.parent.name)])
    if rows:
        show(rows, ["tag", "rank", "alpha", "#modules", "run"])
    return found


# ------------------------------------------------------------ LEAKAGE CHECK
def leakage_check(test_ids):
    """A test file present in training turns 'improvement' into memorisation."""
    train_ids = set()
    for mp in CFG["train_manifests"]:
        mp = Path(mp)
        if not mp.exists():
            continue
        for line in mp.read_text().splitlines():
            if not line.strip():
                continue
            try:
                train_ids.add(json.loads(line).get("source_id", ""))
            except Exception:
                pass
    if not train_ids:
        print("  (no train manifest found - leakage check skipped)")
        return set()
    hits = {t for t in test_ids if t in train_ids}
    if hits:
        print("\n  *** LEAKAGE: these test files are IN THE TRAINING SET ***")
        print("  {}".format(", ".join(sorted(hits))))
        print("  Any WER gain on them is memorisation, not generalisation.")
    else:
        print("  leakage check: clean ({} training sources)".format(len(train_ids)))
    return hits


# ------------------------------------------------------------------- LOAD
def load_test():
    work = CFG["test_dir"]
    audio = {p.stem: p for p in work.rglob("*")
             if p.is_file() and p.suffix.lower() in CFG["audio_exts"]}

    items = []
    for jp in sorted(work.rglob("*.json")):
        if jp.name == "testset_index.json":
            continue
        doc = json.loads(jp.read_text(encoding="utf-8"))
        did = doc.get("id", jp.stem)
        if did not in audio:
            continue
        segs = doc.get("segments", [])
        n = max(len(segs) - 1, 1)
        ov = sum(1 for a, b in zip(segs, segs[1:]) if b["start"] < a["end"]) / n
        verified = bool(doc.get("is_ground_truth") and
                        doc.get("transcript_source") == "manual" and ov < 0.5)
        ref = norm(" ".join(s.get("text", "") for s in segs))
        items.append({"id": did, "audio": audio[did], "ref": ref,
                      "verified": verified, "overlap": round(ov, 2),
                      "words": len(ref.split()),
                      "minutes": doc.get("speech_minutes"),
                      "source": doc.get("transcript_source")})

    show([[i["id"][:22], "VERIFIED" if i["verified"] else "unverified",
           i["source"], i["overlap"], i["minutes"], i["words"],
           sum(term_counts(i["ref"]).values())] for i in items],
         ["id", "tier", "source", "overlap", "min", "words", "terms"])

    leak = leakage_check([i["id"] for i in items])
    for i in items:
        i["leaked"] = i["id"] in leak

    if not CFG["score_unverified"]:
        items = [i for i in items if i["verified"]]
    assert items, "no scorable files"
    return items


# ------------------------------------------------------------------ MODEL
def build_pipe(adapter=None):
    proc = WhisperProcessor.from_pretrained(
        CFG["model"], language="English", task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(
        CFG["model"], dtype=torch.float16, attn_implementation="sdpa")
    if adapter is not None:
        model = PeftModel.from_pretrained(model, str(adapter)).merge_and_unload()

    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.language = CFG["language"]
    model.generation_config.task = CFG["task"]
    model.config.use_cache = True
    model.to("cuda").eval()

    kw = dict(model=model, tokenizer=proc.tokenizer,
              feature_extractor=proc.feature_extractor,
              dtype=torch.float16, device=0)
    if CFG["long_form"] == "chunked":
        kw.update(chunk_length_s=CFG["chunk_len"], batch_size=CFG["batch_size"])
    # "sequential" passes neither -> Whisper's native long-form algorithm
    return pipeline("automatic-speech-recognition", **kw)


def free():
    gc.collect()
    torch.cuda.empty_cache()


# ------------------------------------------------------------------- EVAL
def _bucket(rows):
    if not rows:
        return None
    te = sum(r["errors"] for r in rows)
    tw = sum(r["ref_words"] for r in rows)
    ers = [r["entity_recall"] for r in rows if r["entity_recall"] is not None]
    return {"micro_wer": round(te / tw, 4),
            "macro_wer": round(sum(r["wer"] for r in rows) / len(rows), 4),
            "errors": te, "ref_words": tw,
            "sub": sum(r["sub"] for r in rows),
            "del": sum(r["del"] for r in rows),
            "ins": sum(r["ins"] for r in rows),
            "entity_recall": round(sum(ers) / len(ers), 4) if ers else None,
            "n_files": len(rows)}


def evaluate(tag, adapter, items):
    print("=== {} ===".format(tag), flush=True)
    pipe = build_pipe(adapter)
    per_file = []
    for it in items:
        t0 = time.time()
        gk = {"language": CFG["language"], "task": CFG["task"]}
        hyp = norm(pipe(str(it["audio"]), generate_kwargs=gk)["text"])
        m = jiwer.process_words(it["ref"], hyp)
        er, nt, missed = entity_recall(it["ref"], hyp)
        per_file.append({
            "id": it["id"], "verified": it["verified"], "leaked": it["leaked"],
            "wer": round(m.wer, 4),
            "errors": m.substitutions + m.deletions + m.insertions,
            "ref_words": len(it["ref"].split()),
            "sub": m.substitutions, "del": m.deletions, "ins": m.insertions,
            "entity_recall": er, "n_terms": nt, "missed": missed, "hyp": hyp,
        })
        print("    {:24s} {:10s} WER {:.4f} ({} err)  {:.0f}s".format(
            it["id"][:24], "VERIFIED" if it["verified"] else "unverif",
            m.wer, per_file[-1]["errors"], time.time() - t0), flush=True)
    del pipe
    free()

    ver = [r for r in per_file if r["verified"]]
    unv = [r for r in per_file if not r["verified"]]
    clean = [r for r in ver if not r["leaked"]]
    summary = {"model": tag,
               "verified": _bucket(ver),
               "verified_no_leak": _bucket(clean) if len(clean) != len(ver) else None,
               "unverified": _bucket(unv)}
    if summary["verified"]:
        print("    -> VERIFIED micro {:.4f} | macro {:.4f}\n".format(
            summary["verified"]["micro_wer"], summary["verified"]["macro_wer"]),
            flush=True)
    return per_file, summary


def _table(all_sum, key, title):
    rows = [s for s in all_sum if s.get(key)]
    if not rows:
        return
    base = rows[0][key]
    print("\n{}".format(title))
    show([[s["model"], s[key]["n_files"], s[key]["micro_wer"],
           "-" if i == 0 else "{:+.4f}".format(s[key]["micro_wer"] - base["micro_wer"]),
           s[key]["macro_wer"], s[key]["sub"], s[key]["del"], s[key]["ins"],
           s[key]["entity_recall"]] for i, s in enumerate(rows)],
         ["model", "n", "micro WER", "d micro", "macro WER",
          "sub", "del", "ins", "ent-R"])


def main():
    CFG["out_dir"].mkdir(parents=True, exist_ok=True)
    print("gpu: {} ({:.1f} GB) | long-form: {}\n".format(
        torch.cuda.get_device_name(0),
        torch.cuda.get_device_properties(0).total_memory / 1e9,
        CFG["long_form"]))

    items = load_test()
    print()
    runs = [("baseline", None)] + discover_checkpoints()
    print("\nevaluating {} models on {} files\n".format(len(runs), len(items)))

    all_pf, all_sum = {}, []
    for tag, adapter in runs:
        pf, sm = evaluate(tag, adapter, items)
        all_pf[tag] = pf
        all_sum.append(sm)

    tags = [s["model"] for s in all_sum]
    ids = [r["id"] for r in all_pf[tags[0]]]

    print("\nPER-FILE WER  (* = unverified reference, L = in training set)")
    show([[("{}{}{}".format(i[:20],
            "" if all_pf[tags[0]][k]["verified"] else "*",
            "L" if all_pf[tags[0]][k]["leaked"] else "")),
           all_pf[tags[0]][k]["ref_words"]]
          + [all_pf[t][k]["wer"] for t in tags] for k, i in enumerate(ids)],
         ["file", "words"] + tags)

    print("\nPER-FILE ERROR COUNTS (verified only)")
    show([[i[:20], all_pf[tags[0]][k]["ref_words"]]
          + [all_pf[t][k]["errors"] for t in tags]
          for k, i in enumerate(ids) if all_pf[tags[0]][k]["verified"]],
         ["file", "words"] + tags)

    _table(all_sum, "verified", "GLOBAL - VERIFIED REFERENCES (headline)")
    _table(all_sum, "verified_no_leak", "GLOBAL - VERIFIED, LEAKED FILES REMOVED")
    _table(all_sum, "unverified",
           "GLOBAL - UNVERIFIED REFERENCES (agreement with another ASR, NOT accuracy)")

    ranked = [s for s in all_sum if s.get("verified")]
    best = min(ranked, key=lambda s: s["verified"]["micro_wer"])
    print("\nbest VERIFIED micro WER: {} ({:.4f})".format(
        best["model"], best["verified"]["micro_wer"]))

    # concentration check: is the whole gain from one file?
    if best["model"] != "baseline":
        b = {r["id"]: r["errors"] for r in all_pf["baseline"]}
        d = [(r["id"], b[r["id"]] - r["errors"])
             for r in all_pf[best["model"]] if r["verified"]]
        gains = sorted([x for x in d if x[1] > 0], key=lambda x: -x[1])
        tot = sum(x[1] for x in d)
        if gains and tot > 0 and gains[0][1] / max(tot, 1) > 0.8:
            print("\n  NOTE: {:.0f}% of the improvement comes from a single file ({}).".format(
                100 * gains[0][1] / tot, gains[0][0]))
            print("  Domain adaptation lifts many files a little. A gain concentrated")
            print("  in one file is more likely leakage or a lucky match - verify that")
            print("  file is not in the training set before reporting this as a win.")

    (CFG["out_dir"] / "wer_results.json").write_text(json.dumps(
        {"summary": all_sum,
         "per_file": {t: [{k: v for k, v in r.items() if k != "hyp"}
                          for r in all_pf[t]] for t in tags}}, indent=2))
    with open(CFG["out_dir"] / "wer_diffs.txt", "w", encoding="utf-8") as fh:
        for k, i in enumerate(ids):
            fh.write("=" * 78 + "\n{}  [{}]\n".format(
                i, "VERIFIED" if all_pf[tags[0]][k]["verified"] else "UNVERIFIED"))
            fh.write("REF   : {}\n".format(items[k]["ref"][:2500]))
            for t in tags:
                fh.write("{:12s}: {}\n".format(t, all_pf[t][k]["hyp"][:2500]))
            fh.write("\n")
    print("\nwrote {} and wer_diffs.txt".format(
        CFG["out_dir"] / "wer_results.json"))


if __name__ == "__main__":
    main()

gpu: Tesla T4 (16.7 GB) | long-form: chunked



| id          | tier       | source    |   overlap |   min |   words |   terms |
|-------------|------------|-----------|-----------|-------|---------|---------|
| 67Ljjk8Blfc | unverified | generated |      0.97 |  11.5 |    1014 |       7 |
| Wuu3Gd4Q6Uo | VERIFIED   | manual    |      0    |  10   |    2752 |      79 |
| lLnHN0IWic0 | VERIFIED   | manual    |      0    |   4.3 |     914 |      13 |
| mYmNM8-XRP0 | VERIFIED   | manual    |      0.01 |   5.2 |     921 |      18 |
| yR6pLfgEK3U | VERIFIED   | manual    |      0    |   2.3 |     528 |       4 |
| zQvFSGxDiBI | VERIFIED   | manual    |      0    |   3   |     601 |       2 |
  (no train manifest found - leakage check skipped)

| tag                |   rank |   alpha |   #modules | run              |
|--------------------|--------|---------|------------|------------------|
| r512:40            |    512 |    1024 |          6 | whisper_lora_512 |
| r64:80             |     64 |     128 |          6 | whisper_lora_64  |
| r

Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


    67Ljjk8Blfc              unverif    WER 0.0286 (29 err)  13s
    Wuu3Gd4Q6Uo              VERIFIED   WER 0.0269 (74 err)  29s
    lLnHN0IWic0              VERIFIED   WER 0.0142 (13 err)  11s
    mYmNM8-XRP0              VERIFIED   WER 0.0217 (20 err)  12s
    yR6pLfgEK3U              VERIFIED   WER 0.1042 (55 err)  8s
    zQvFSGxDiBI              VERIFIED   WER 0.0449 (27 err)  8s
    -> VERIFIED micro 0.0331 | macro 0.0424

=== r512:40 ===


Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


...

    67Ljjk8Blfc              unverif    WER 0.0266 (27 err)  13s
    Wuu3Gd4Q6Uo              VERIFIED   WER 0.0287 (79 err)  31s
    lLnHN0IWic0              VERIFIED   WER 0.0153 (14 err)  11s
    mYmNM8-XRP0              VERIFIED   WER 0.0217 (20 err)  13s
    yR6pLfgEK3U              VERIFIED   WER 0.1080 (57 err)  8s
    zQvFSGxDiBI              VERIFIED   WER 0.0433 (26 err)  9s
    -> VERIFIED micro 0.0343 | macro 0.0434

=== whisper_lora:140 ===


Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


    67Ljjk8Blfc              unverif    WER 0.0276 (28 err)  14s
    Wuu3Gd4Q6Uo              VERIFIED   WER 0.0294 (81 err)  31s
    lLnHN0IWic0              VERIFIED   WER 0.0153 (14 err)  11s
    mYmNM8-XRP0              VERIFIED   WER 0.0228 (21 err)  13s
    yR6pLfgEK3U              VERIFIED   WER 0.1098 (58 err)  8s
    zQvFSGxDiBI              VERIFIED   WER 0.0466 (28 err)  9s
    -> VERIFIED micro 0.0353 | macro 0.0448

=== whisper_lora:final ===


Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


    67Ljjk8Blfc              unverif    WER 0.0266 (27 err)  14s
    Wuu3Gd4Q6Uo              VERIFIED   WER 0.0280 (77 err)  31s
    lLnHN0IWic0              VERIFIED   WER 0.0142 (13 err)  11s
    mYmNM8-XRP0              VERIFIED   WER 0.0217 (20 err)  13s
    yR6pLfgEK3U              VERIFIED   WER 0.1080 (57 err)  8s
    zQvFSGxDiBI              VERIFIED   WER 0.0466 (28 err)  9s
    -> VERIFIED micro 0.0341 | macro 0.0437


PER-FILE WER  (* = unverified reference, L = in training set)
| file         |   words |   baseline |   r512:40 |   r64:80 |   r64:120 |   r64:140 |   r64:final |   whisper_lora:80 |   whisper_lora:120 |   whisper_lora:140 |   whisper_lora:final |
|--------------|---------|------------|-----------|----------|-----------|-----------|-------------|-------------------|--------------------|--------------------|----------------------|
| 67Ljjk8Blfc* |    1014 |     0.0286 |    0.0276 |   0.0237 |    0.0256 |    0.0266 |      0.0237 |            0.0266 |            

In [6]:
#!/usr/bin/env python3
"""
win_loss_diff.py
----------------
Find the exact reference positions where model B beats model A and vice versa.

    python -u win_loss_diff.py

Reads hyp_cache.json written by insertion_diff.py (no GPU needed if cached).

----------------------------------------------------------------------------
HOW IT WORKS
----------------------------------------------------------------------------
Each hypothesis is aligned to the SAME reference, giving a map:
    ref word index -> what that model produced there (or <DEL>)
Comparing the two maps at every index yields four cases:

    WIN    A wrong, B correct        <- the examples you want to show
    LOSS   A correct, B wrong        <- report these too, or the slide is dishonest
    BOTH   both wrong (same or not)
    TIE    both correct

A fine-tune that genuinely improved recognition produces many WINs and few
LOSSes. If WINs and LOSSes are roughly balanced, the model did not get better
at hearing - it just moved errors around, and any WER delta came from
insertions/loops instead.
----------------------------------------------------------------------------
"""

import json
import re
import unicodedata
from pathlib import Path

import jiwer
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None

BASE = Path("/home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning")

CFG = {
    "test_dir": BASE / "test_set",
    "out_dir":  BASE / "eval_out",
    "cache":    BASE / "eval_out" / "hyp_cache.json",
    "model_a":  "baseline",
    "model_b":  "r64:80",
    "focus":    "Wuu3Gd4Q6Uo",   # None = all files
    "context":  8,               # ref words shown either side
    "max_show": 25,
}

normalizer = EnglishTextNormalizer({})


def show(rows, headers):
    if tabulate:
        print(tabulate(rows, headers=headers, tablefmt="github"), flush=True)
    else:
        print("  ".join(headers))
        for r in rows:
            print("  ".join(str(x) for x in r))


def norm(t):
    t = unicodedata.normalize("NFKC", t).replace("-", " ")
    return re.sub(r"\s+", " ", normalizer(t)).strip()


def load_refs():
    refs = {}
    for jp in sorted(CFG["test_dir"].rglob("*.json")):
        if jp.name in ("testset_index.json", "hyp_cache.json",
                       "insertion_report.json"):
            continue
        doc = json.loads(jp.read_text(encoding="utf-8"))
        segs = doc.get("segments", [])
        n = max(len(segs) - 1, 1)
        ov = sum(1 for a, b in zip(segs, segs[1:]) if b["start"] < a["end"]) / n
        if not (doc.get("is_ground_truth") and
                doc.get("transcript_source") == "manual" and ov < 0.5):
            continue
        refs[doc.get("id", jp.stem)] = norm(
            " ".join(s.get("text", "") for s in segs))
    return refs


def ref_map(ref, hyp):
    """ref word index -> string the model produced there.

    'equal'/'substitute' map 1:1 onto ref indices; 'delete' means the model
    produced nothing there. Insertions are ignored here - they belong to no ref
    index and are covered by insertion_diff.py.
    """
    out = jiwer.process_words(ref, hyp)
    rw, hw = ref.split(), hyp.split()
    m = {}
    for ch in out.alignments[0]:
        if ch.type == "equal":
            for k in range(ch.ref_end_idx - ch.ref_start_idx):
                m[ch.ref_start_idx + k] = hw[ch.hyp_start_idx + k]
        elif ch.type == "substitute":
            n = ch.ref_end_idx - ch.ref_start_idx
            for k in range(n):
                hi = ch.hyp_start_idx + k
                m[ch.ref_start_idx + k] = hw[hi] if hi < ch.hyp_end_idx else "<DEL>"
        elif ch.type == "delete":
            for k in range(ch.ref_end_idx - ch.ref_start_idx):
                m[ch.ref_start_idx + k] = "<DEL>"
    for i in range(len(rw)):
        m.setdefault(i, "<DEL>")
    return m, out


def band(rw, i, w):
    lo, hi = max(i - w, 0), min(i + w + 1, len(rw))
    return " ".join(rw[lo:i]), " ".join(rw[i + 1:hi])


def main():
    cache = json.loads(CFG["cache"].read_text())
    a, b = CFG["model_a"], CFG["model_b"]
    refs = load_refs()
    ids = [CFG["focus"]] if CFG["focus"] else sorted(refs)

    grand = {"win": 0, "loss": 0, "both": 0}
    for fid in ids:
        if fid not in refs:
            print("no verified reference for {}".format(fid))
            continue
        ref = refs[fid]
        rw = ref.split()
        ma, oa = ref_map(ref, cache[a][fid])
        mb, ob = ref_map(ref, cache[b][fid])

        wins, losses, boths = [], [], []
        for i, w in enumerate(rw):
            ca, cb = ma[i] == w, mb[i] == w
            if ca and cb:
                continue
            if not ca and cb:
                wins.append(i)
            elif ca and not cb:
                losses.append(i)
            else:
                boths.append(i)

        print("\n" + "=" * 78)
        print("{}   ({} ref words)".format(fid, len(rw)))
        print("=" * 78)
        show([[a, oa.substitutions, oa.deletions, oa.insertions],
              [b, ob.substitutions, ob.deletions, ob.insertions]],
             ["model", "sub", "del", "ins"])
        print("\n  {} WINS  ({} wrong -> {} right)".format(len(wins), a, b))
        print("  {} LOSSES ({} right -> {} wrong)".format(len(losses), a, b))
        print("  {} both wrong".format(len(boths)))

        grand["win"] += len(wins)
        grand["loss"] += len(losses)
        grand["both"] += len(boths)

        for title, idx in (("WINS  ({} better)".format(b), wins),
                           ("LOSSES ({} worse)".format(b), losses)):
            print("\n  --- {} ---".format(title))
            if not idx:
                print("    (none)")
            for i in idx[:CFG["max_show"]]:
                pre, post = band(rw, i, CFG["context"])
                print("\n    [{}] ...{}  __  {}...".format(i, pre[-60:], post[:60]))
                print("      REF : {}".format(rw[i]))
                print("      {:8s}: {}".format(a, ma[i]))
                print("      {:8s}: {}".format(b, mb[i]))

        # multi-word runs read better on a slide than isolated words
        runs = []
        cur = []
        for i in wins:
            if cur and i == cur[-1] + 1:
                cur.append(i)
            else:
                if len(cur) > 1:
                    runs.append(cur)
                cur = [i]
        if len(cur) > 1:
            runs.append(cur)
        if runs:
            print("\n  --- CONSECUTIVE WIN RUNS (best slide material) ---")
            for r in runs:
                pre, post = band(rw, r[0], CFG["context"])
                print("\n    ref[{}-{}]  ...{}".format(r[0], r[-1], pre[-70:]))
                print("      REF : {}".format(" ".join(rw[i] for i in r)))
                print("      {:8s}: {}".format(a, " ".join(ma[i] for i in r)))
                print("      {:8s}: {}".format(b, " ".join(mb[i] for i in r)))

    print("\n" + "=" * 78)
    print("TOTAL   wins {}  losses {}  both-wrong {}".format(
        grand["win"], grand["loss"], grand["both"]))
    n = grand["win"] + grand["loss"]
    if n:
        print("net recognition change: {:+d} words".format(grand["win"] - grand["loss"]))
        if abs(grand["win"] - grand["loss"]) <= max(2, 0.2 * n):
            print("\n  Wins and losses are roughly balanced -> the fine-tune did NOT")
            print("  improve recognition; it redistributed errors. Any WER gain came")
            print("  from insertions/loops, not from hearing words more accurately.")
            print("  Show the loop as the result; do not present a cherry-picked win")
            print("  as evidence of domain adaptation.")


if __name__ == "__main__":
    main()


Wuu3Gd4Q6Uo   (2752 ref words)
| model    |   sub |   del |   ins |
|----------|-------|-------|-------|
| baseline |    16 |     9 |    49 |
| r64:80   |    18 |    10 |     4 |

  4 WINS  (baseline wrong -> r64:80 right)
  7 LOSSES (baseline right -> r64:80 wrong)
  21 both wrong

  --- WINS  (r64:80 better) ---

    [1580] ...this has some give to it it distributes  __  force over the sandpaper a little bit better...
      REF : that
      baseline: <DEL>
      r64:80  : that

    [1685] ...all gummed up just flick it a couple  __  times and you can see the dust comes...
      REF : of
      baseline: couple
      r64:80  : of

    [1847] ...have bare metal as well now we also  __  into the surrounding paintwork not a lot but...
      REF : sanded
      baseline: it
      r64:80  : sanded

    [2277] ...much easier every can of filler comes with  __  sheet and makes it impossible to mess up...
      REF : the
      baseline: this
      r64:80  : the

  --- LOSSES (r64:80 worse) ---

In [ ]:
#!/usr/bin/env python3
"""
val_eval.py
-----------
WER on the VALIDATION split, read directly from the featurized HF dataset.

    python -u val_eval.py

The split is an Arrow dataset of precomputed features, not audio:
    input_features : (128, 3000) log-mel   <- feed straight to generate()
    labels         : token ids             <- decode back for the reference

No feature extraction needed, so this is much faster than the test-set run.

----------------------------------------------------------------------------
WHAT THIS MEASURES  -  READ BEFORE QUOTING ANY NUMBER
----------------------------------------------------------------------------
The val references are NOT ground truth. They are Whisper large-v3 output with
LLM correction - the same pipeline that produced the TRAINING labels, from the
same 86 videos, split by source_id.

    val WER = AGREEMENT WITH THE TRAINING LABEL DISTRIBUTION

A fine-tuned model scores well here almost by construction: reproducing this
text WAS the training objective. A large val gain is NOT evidence of better
transcription.

The value is the CONTRAST with held-out test WER:
    big val gain + flat test gain -> learned the pseudo-labels, not the domain
    val gain ~= test gain         -> the labels carried real signal

This split also drove early stopping, so it is model-SELECTION data, not
held-out data. Training diagnostic only - never a reported result.
----------------------------------------------------------------------------
"""

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import gc
import json
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import torch
import jiwer
from datasets import load_from_disk
from peft import PeftModel
from transformers import (WhisperForConditionalGeneration, WhisperProcessor)
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None

BASE = Path("/home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning")

CFG = {
    "model":   "openai/whisper-large-v3-turbo",
    # point at the DatasetDict root, or directly at the validation dir
    "ds_path": BASE / "stt_audio" / "whisper_ds",
    "split":   "validation",
    "out_dir": BASE / "eval_out",

    "checkpoints": {
        "r512:40": BASE / "stt_audio" / "whisper_lora_512" / "checkpoint-40",
        "r64:80":  BASE / "stt_audio" / "whisper_lora_64" / "checkpoint-80",
        "r16:80":  BASE / "stt_audio" / "whisper_lora" / "checkpoint-80",
    },

    # test-set micro WER from wer_eval, for the contrast table
    "test_micro": {"baseline": 0.0331, "r512:40": 0.0257,
                   "r64:80": 0.0259, "r16:80": 0.0341},

    "batch_size":   16,
    "max_new":      440,     # Whisper decoder limit is 448
    "max_items":    None,    # e.g. 200 for a quick pass
    "num_beams":    1,       # greedy: matches training-time decoding
}

normalizer = EnglishTextNormalizer({})


def show(rows, headers):
    if tabulate:
        print(tabulate(rows, headers=headers, tablefmt="github"), flush=True)
    else:
        print("  ".join(headers))
        for r in rows:
            print("  ".join(str(x) for x in r))


def norm(t):
    t = unicodedata.normalize("NFKC", t).replace("-", " ")
    return re.sub(r"\s+", " ", normalizer(t)).strip()


# ------------------------------------------------------------------- LOAD
def load_val(tokenizer):
    p = CFG["ds_path"]
    ds = load_from_disk(str(p))
    if hasattr(ds, "keys"):                       # a DatasetDict
        assert CFG["split"] in ds, "split '{}' not in {}".format(
            CFG["split"], list(ds.keys()))
        ds = ds[CFG["split"]]
    if CFG["max_items"]:
        ds = ds.select(range(min(CFG["max_items"], len(ds))))

    print("columns: {}".format(ds.column_names))

    # reference text: prefer a kept 'text' column, else decode the labels
    if "text" in ds.column_names:
        refs = [norm(t) for t in ds["text"]]
        src = "text column"
    else:
        lab = []
        for l in ds["labels"]:
            ids = [i for i in l if i != -100]
            lab.append(ids)
        refs = [norm(t) for t in
                tokenizer.batch_decode(lab, skip_special_tokens=True)]
        src = "decoded from labels"

    feat = np.asarray(ds[0]["input_features"], dtype=np.float32)
    show([["chunks", len(ds)],
          ["reference from", src],
          ["mel shape", feat.shape],
          ["ref words", sum(len(r.split()) for r in refs)],
          ["empty refs", sum(1 for r in refs if len(r.split()) < 3)]],
         ["val split", "value"])

    keep = [i for i, r in enumerate(refs) if len(r.split()) >= 3]
    if len(keep) < len(refs):
        print("  dropping {} chunks with <3 reference words".format(
            len(refs) - len(keep)))
        ds = ds.select(keep)
        refs = [refs[i] for i in keep]
    return ds, refs


# ------------------------------------------------------------------ MODEL
def build_model(adapter):
    model = WhisperForConditionalGeneration.from_pretrained(
        CFG["model"], dtype=torch.float16, attn_implementation="sdpa")
    if adapter is not None:
        assert Path(adapter).exists(), "adapter not found: {}".format(adapter)
        model = PeftModel.from_pretrained(model, str(adapter)).merge_and_unload()
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.language = "en"
    model.generation_config.task = "transcribe"
    model.config.use_cache = True          # trainer disabled it for checkpointing
    return model.to("cuda").eval()


@torch.no_grad()
def transcribe(model, tokenizer, ds):
    hyps = []
    n, bs = len(ds), CFG["batch_size"]
    for i in range(0, n, bs):
        chunk = ds[i:i + bs]["input_features"]
        feats = torch.tensor(np.asarray(chunk, dtype=np.float32),
                             dtype=torch.float16, device="cuda")
        ids = model.generate(feats, max_new_tokens=CFG["max_new"],
                             num_beams=CFG["num_beams"], language="en",
                             task="transcribe")
        hyps += [norm(t) for t in
                 tokenizer.batch_decode(ids, skip_special_tokens=True)]
        if (i // bs) % 10 == 0:
            print("    {}/{}".format(min(i + bs, n), n), flush=True)
    return hyps


def evaluate(tag, adapter, ds, refs, tokenizer):
    print("\n=== {} ===".format(tag), flush=True)
    t0 = time.time()
    model = build_model(adapter)
    hyps = transcribe(model, tokenizer, ds)
    del model
    gc.collect()
    torch.cuda.empty_cache()

    m = jiwer.process_words(refs, hyps)
    errs = m.substitutions + m.deletions + m.insertions
    words = sum(len(r.split()) for r in refs)
    exact = sum(1 for r, h in zip(refs, hyps) if r == h)

    s = {"model": tag,
         "micro_wer": round(errs / words, 4),
         "sub": m.substitutions, "del": m.deletions, "ins": m.insertions,
         "errors": errs, "words": words,
         "exact_match": round(exact / len(refs), 4),
         "minutes": round((time.time() - t0) / 60, 1)}
    print("    val WER {:.4f} | exact-match {:.1%} | {:.1f} min".format(
        s["micro_wer"], s["exact_match"], s["minutes"]), flush=True)
    return s, hyps


def main():
    CFG["out_dir"].mkdir(parents=True, exist_ok=True)
    print("gpu: {}\n".format(torch.cuda.get_device_name(0)))

    processor = WhisperProcessor.from_pretrained(
        CFG["model"], language="English", task="transcribe")
    tokenizer = processor.tokenizer

    ds, refs = load_val(tokenizer)

    runs = [("baseline", None)] + list(CFG["checkpoints"].items())
    summaries, all_hyps = [], {}
    for tag, adapter in runs:
        s, h = evaluate(tag, adapter, ds, refs, tokenizer)
        summaries.append(s)
        all_hyps[tag] = h

    base = summaries[0]
    print("\n\nVALIDATION (references = Whisper + LLM correction, NOT truth)")
    show([[s["model"], s["micro_wer"],
           "-" if s is base else "{:+.4f}".format(s["micro_wer"] - base["micro_wer"]),
           "-" if s is base else "{:+.1f}%".format(
               100 * (s["micro_wer"] - base["micro_wer"]) / base["micro_wer"]),
           s["sub"], s["del"], s["ins"], "{:.1%}".format(s["exact_match"])]
          for s in summaries],
         ["model", "val WER", "delta", "rel", "sub", "del", "ins", "exact"])

    print("\n\nVAL vs TEST  (the diagnostic)")
    tm = CFG["test_micro"]
    rows = []
    for s in summaries:
        t = tm.get(s["model"])
        vrel = 0.0 if s is base else \
            100 * (s["micro_wer"] - base["micro_wer"]) / base["micro_wer"]
        trel = 0.0 if (s is base or t is None) else \
            100 * (t - tm["baseline"]) / tm["baseline"]
        rows.append([s["model"], s["micro_wer"],
                     "-" if s is base else "{:+.1f}%".format(vrel),
                     t if t is not None else "-",
                     "-" if (s is base or t is None) else "{:+.1f}%".format(trel)])
    show(rows, ["model", "val WER", "val rel", "test WER", "test rel"])

    if len(summaries) > 1:
        best = min(summaries[1:], key=lambda s: s["micro_wer"])
        vrel = 100 * (best["micro_wer"] - base["micro_wer"]) / base["micro_wer"]
        t = tm.get(best["model"])
        trel = 100 * (t - tm["baseline"]) / tm["baseline"] if t else None
        print("\nbest val: {}  ({:+.1f}% val, {} test)".format(
            best["model"], vrel,
            "{:+.1f}%".format(trel) if trel is not None else "n/a"))
        if trel is not None and vrel < -10 and trel > -10:
            print("""
  -> Large val gain, flat test gain. The model learned the PSEUDO-LABEL
     DISTRIBUTION, not the domain. Val references came from Whisper + LLM
     correction, so reproducing them was the training objective - not
     evidence of better transcription. Quote the val number only next to
     the test number, labelled as agreement-with-training-labels.""")
        elif trel is not None and abs(vrel - trel) < 5:
            print("""
  -> Val and test moved together: the labels carried real signal and the
     gain is likely genuine. Worth extending the training set.""")

    (CFG["out_dir"] / "val_results.json").write_text(
        json.dumps({"summary": summaries}, indent=2))

    if len(summaries) > 1:
        h = all_hyps[best["model"]]
        worst = sorted(((jiwer.wer(r, hh), r, hh) for r, hh in zip(refs, h)
                        if len(r.split()) >= 8), key=lambda x: -x[0])[:12]
        with open(CFG["out_dir"] / "val_worst.txt", "w", encoding="utf-8") as fh:
            for w, r, hh in worst:
                fh.write("WER {:.2f}\nREF: {}\nHYP: {}\n\n".format(w, r, hh))
        print("\nwrote val_results.json and val_worst.txt")


if __name__ == "__main__":
    main()

In [10]:
#!/usr/bin/env python3
"""Baseline vs best checkpoint - WER and error decomposition."""

from tabulate import tabulate

BASELINE = "baseline"
BEST     = "r64:80"

DATA = [
    # (metric,           baseline, best,  is_wer)
    ("TEST  micro WER",    0.0331, 0.0259, True),
    ("TEST  substitutions",    56,     59, False),
    ("TEST  deletions",        23,     24, False),
    ("TEST  insertions",      110,     65, False),
    ("VAL   micro WER",    0.0736, 0.0629, True),
    ("VAL   substitutions",   205,    211, False),
    ("VAL   deletions",       484,    314, False),
    ("VAL   insertions",      143,    186, False),
]


def pct(new, old):
    return "-" if not old else "{:+.1f}%".format(100.0 * (new - old) / old)


def fmt(v, is_wer):
    return "{:.4f}".format(v) if is_wer else "{:d}".format(v)


rows = [[m, fmt(b, w), fmt(n, w), pct(n, b)] for m, b, n, w in DATA]

print(tabulate(rows,
               headers=["metric", BASELINE, BEST, "% change"],
               tablefmt="github"))

| metric              |   baseline |   r64:80 | % change   |
|---------------------|------------|----------|------------|
| TEST  micro WER     |     0.0331 |   0.0259 | -21.8%     |
| TEST  substitutions |    56      |  59      | +5.4%      |
| TEST  deletions     |    23      |  24      | +4.3%      |
| TEST  insertions    |   110      |  65      | -40.9%     |
| VAL   micro WER     |     0.0736 |   0.0629 | -14.5%     |
| VAL   substitutions |   205      | 211      | +2.9%      |
| VAL   deletions     |   484      | 314      | -35.1%     |
| VAL   insertions    |   143      | 186      | +30.1%     |
